In [2]:
import urllib.request

# Configuration
dataset_name = "ruoka"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

# Extract GT keywords and store in a set
gt_keywords = set()

print(f"Extracting GT keywords from {dataset_name} dataset...")
for i in range(num_pages):
    try:
        gt_url = f"{base_url}/{i}/GT.txt"
        with urllib.request.urlopen(gt_url, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            keywords = gt_text.lower().split()
            gt_keywords.update(keywords)
    except:
        pass

print(f"✓ Extracted {len(gt_keywords)} unique GT keywords")
print(f"\nFirst 20 keywords: {list(gt_keywords)[:20]}")

Extracting GT keywords from ruoka dataset...
✓ Extracted 346 unique GT keywords

First 20 keywords: ['rieska', 'voileipäkakku', 'paistisuikaleet', 'faustino', 'lehtikaali', 'lisukke', 'gluteenito', 'kurpitsa', 'joulu', 'sekavihannekset', 'kuohuviinit', 'kotilääkäri', 'munakkaat', 'pestot', 'teeleipä', 'riisipallot', 'frozen', 'nutella', 'leipä', 'siikafileet']


## Step 2: Apply Stemming

In [3]:
from nltk.stem.snowball import SnowballStemmer
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Initialize Finnish stemmer (ruoka dataset is Finnish)
stemmer = SnowballStemmer('finnish')

# Apply stemming
stemmed_keywords = {}
changed_count = 0

for keyword in gt_keywords:
    stemmed = stemmer.stem(keyword)
    stemmed_keywords[keyword] = stemmed
    if keyword != stemmed:
        changed_count += 1

# Show only changed keywords
print("STEMMING RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by stemming: {changed_count}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count}")
print(f"\nChanged keywords (Original → Stemmed):")
print("-" * 60)

changed_keywords = [(orig, stemmed) for orig, stemmed in stemmed_keywords.items() if orig != stemmed]
for i, (orig, stemmed) in enumerate(sorted(changed_keywords)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {stemmed}")

if len(changed_keywords) > 30:
    print(f"... and {len(changed_keywords) - 30} more changed keywords")

STEMMING RESULTS
Total keywords: 346
Keywords changed by stemming: 266
Keywords unchanged: 80

Changed keywords (Original → Stemmed):
------------------------------------------------------------
  1. aamiaine             → aamia
  2. aasialainen          → aasialain
  3. afrika               → afrik
  4. ankanrintaa          → ankanrint
  5. anna                 → an
  6. annosfondi           → annosfond
  7. appelsiinikastiketta → appelsiinikastik
  8. arancini             → aranc
  9. arkiruoat            → arkiruoa
 10. basmati              → basmat
 11. bataatti             → bataat
 12. broileri             → broiler
 13. broilerin            → broiler
 14. broilerinliha        → broilerinlih
 15. broileriruoat        → broileriruoa
 16. buffet               → buf
 17. carpaccio            → carpacio
 18. chicken              → chick
 19. couscoussalaatti     → couscoussalaat
 20. curry                → cury
 21. curryt               → cury
 22. espanja              → espanj
 23. 

## Step 3: Apply Lemmatization

In [4]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download required NLTK data for lemmatization
for package in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger']:
    try:
        nltk.data.find(f'corpora/{package}')
    except LookupError:
        try:
            nltk.download(package, quiet=True)
        except:
            pass

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Apply lemmatization (using noun as default POS)
lemmatized_keywords = {}
changed_count_lemma = 0

for keyword in gt_keywords:
    # Try lemmatizing as noun, verb, and adjective, pick the shortest
    lemma_noun = lemmatizer.lemmatize(keyword, pos='n')
    lemma_verb = lemmatizer.lemmatize(keyword, pos='v')
    lemma_adj = lemmatizer.lemmatize(keyword, pos='a')
    
    # Pick the shortest lemma (most reduced form)
    lemmatized = min([lemma_noun, lemma_verb, lemma_adj], key=len)
    lemmatized_keywords[keyword] = lemmatized
    
    if keyword != lemmatized:
        changed_count_lemma += 1

# Show only changed keywords
print("LEMMATIZATION RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by lemmatization: {changed_count_lemma}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count_lemma}")
print(f"\nChanged keywords (Original → Lemmatized):")
print("-" * 60)

changed_keywords_lemma = [(orig, lemma) for orig, lemma in lemmatized_keywords.items() if orig != lemma]
for i, (orig, lemma) in enumerate(sorted(changed_keywords_lemma)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {lemma}")

if len(changed_keywords_lemma) > 30:
    print(f"... and {len(changed_keywords_lemma) - 30} more changed keywords")

LEMMATIZATION RESULTS
Total keywords: 346
Keywords changed by lemmatization: 1
Keywords unchanged: 345

Changed keywords (Original → Lemmatized):
------------------------------------------------------------
  1. tapas                → tapa


## Step 4: Compare Stemming vs Lemmatization

In [5]:
# Compare stemming vs lemmatization
print("COMPARISON: STEMMING vs LEMMATIZATION")
print("=" * 70)
print(f"{'Metric':<40s} {'Stemming':>12s} {'Lemmatization':>15s}")
print("-" * 70)
print(f"{'Total keywords':<40s} {len(gt_keywords):>12d} {len(gt_keywords):>15d}")
print(f"{'Keywords changed':<40s} {changed_count:>12d} {changed_count_lemma:>15d}")
print(f"{'Keywords unchanged':<40s} {len(gt_keywords)-changed_count:>12d} {len(gt_keywords)-changed_count_lemma:>15d}")
print(f"{'Change rate (%)':<40s} {changed_count/len(gt_keywords)*100:>11.1f}% {changed_count_lemma/len(gt_keywords)*100:>14.1f}%")
print("=" * 70)

# Find keywords where stemming and lemmatization differ
different_results = []
for keyword in gt_keywords:
    if stemmed_keywords[keyword] != lemmatized_keywords[keyword]:
        different_results.append((keyword, stemmed_keywords[keyword], lemmatized_keywords[keyword]))

print(f"\nKeywords with different results (Stemming ≠ Lemmatization): {len(different_results)}")
if different_results:
    print("\nExamples (Original → Stemmed → Lemmatized):")
    print("-" * 70)
    for i, (orig, stem, lemma) in enumerate(sorted(different_results)[:20], 1):
        print(f"{i:3d}. {orig:15s} → {stem:15s} → {lemma}")
    
    if len(different_results) > 20:
        print(f"... and {len(different_results) - 20} more differences")

COMPARISON: STEMMING vs LEMMATIZATION
Metric                                       Stemming   Lemmatization
----------------------------------------------------------------------
Total keywords                                    346             346
Keywords changed                                  266               1
Keywords unchanged                                 80             345
Change rate (%)                                 76.9%            0.3%

Keywords with different results (Stemming ≠ Lemmatization): 267

Examples (Original → Stemmed → Lemmatized):
----------------------------------------------------------------------
  1. aamiaine        → aamia           → aamiaine
  2. aasialainen     → aasialain       → aasialainen
  3. afrika          → afrik           → afrika
  4. ankanrintaa     → ankanrint       → ankanrintaa
  5. anna            → an              → anna
  6. annosfondi      → annosfond       → annosfondi
  7. appelsiinikastiketta → appelsiinikastik → appelsiinik

## Step 5: POS Analysis of Stemmed Keywords

In [6]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from collections import Counter

# Download required NLTK data for POS tagging
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

# Get unique stemmed keywords
stemmed_unique = set(stemmed_keywords.values())
print(f"Total unique stemmed keywords: {len(stemmed_unique)}")
print(f"Original keywords: {len(gt_keywords)}")
print(f"Reduction from stemming: {len(gt_keywords) - len(stemmed_unique)} keywords merged\n")

# POS tag all stemmed keywords
pos_tagged = {}
pos_distribution = Counter()

print("Performing POS tagging on stemmed keywords...")
for stemmed_keyword in stemmed_unique:
    try:
        tokens = word_tokenize(stemmed_keyword)
        tags = pos_tag(tokens, tagset='universal')
        # Use the POS tag of the first token (most common in single-word keywords)
        pos = tags[0][1] if tags else 'UNK'
        pos_tagged[stemmed_keyword] = pos
        pos_distribution[pos] += 1
    except:
        pos_tagged[stemmed_keyword] = 'UNK'
        pos_distribution['UNK'] += 1

# POS mapping for readable labels
pos_labels = {
    'NOUN': 'Noun',
    'VERB': 'Verb',
    'ADJ': 'Adjective',
    'ADV': 'Adverb',
    'PRON': 'Pronoun',
    'ADP': 'Adposition',
    'CCONJ': 'Coord. Conjunction',
    'SCONJ': 'Subord. Conjunction',
    'DET': 'Determiner',
    'NUM': 'Number',
    'INTJ': 'Interjection',
    'PART': 'Particle',
    'PROPN': 'Proper Noun',
    'SYM': 'Symbol',
    'X': 'Other',
    'UNK': 'Unknown',
    '.': 'Punctuation'
}

print("\n" + "=" * 60)
print("POS DISTRIBUTION OF STEMMED KEYWORDS")
print("=" * 60)

# Sort by frequency
for pos, count in pos_distribution.most_common():
    label = pos_labels.get(pos, pos)
    percentage = (count / len(stemmed_unique)) * 100
    print(f"{label:20s}: {count:4d} ({percentage:6.2f}%)")

print("=" * 60)

# Show examples for each POS
print("\nEXAMPLES BY POS (first 5 of each):\n")

for pos in sorted(pos_distribution.keys(), key=lambda x: pos_distribution[x], reverse=True):
    keywords_with_pos = [k for k, p in pos_tagged.items() if p == pos]
    label = pos_labels.get(pos, pos)
    print(f"{label} ({len(keywords_with_pos)} keywords):")
    for keyword in sorted(keywords_with_pos)[:5]:
        print(f"  - {keyword}")
    print()

Total unique stemmed keywords: 332
Original keywords: 346
Reduction from stemming: 14 keywords merged

Performing POS tagging on stemmed keywords...

POS DISTRIBUTION OF STEMMED KEYWORDS
Noun                :  329 ( 99.10%)
Verb                :    2 (  0.60%)
Determiner          :    1 (  0.30%)

EXAMPLES BY POS (first 5 of each):

Noun (329 keywords):
  - aamia
  - aasialain
  - afrik
  - alkoholito
  - ankanrint

Verb (2 keywords):
  - can
  - kananuget

Determiner (1 keywords):
  - an



## Step 6: Character Length Analysis of Stemmed Keywords

In [9]:
import numpy as np

# Analyze character length of stemmed keywords
stemmed_list = list(stemmed_unique)
char_lengths = [len(keyword) for keyword in stemmed_list]

# Calculate statistics
length_stats = {
    'mean': np.mean(char_lengths),
    'median': np.median(char_lengths),
    'min': np.min(char_lengths),
    'max': np.max(char_lengths),
    'std': np.std(char_lengths),
    '25th_percentile': np.percentile(char_lengths, 25),
    '75th_percentile': np.percentile(char_lengths, 75)
}

print("\n" + "=" * 60)
print("CHARACTER LENGTH ANALYSIS - STEMMED KEYWORDS")
print("=" * 60)
print(f"Total unique stemmed keywords: {len(stemmed_list)}")
print(f"\nStatistics:")
print(f"  Mean length:         {length_stats['mean']:.2f} characters")
print(f"  Median length:       {length_stats['median']:.0f} characters")
print(f"  Min length:          {length_stats['min']} characters")
print(f"  Max length:          {length_stats['max']} characters")
print(f"  Std Dev:             {length_stats['std']:.2f}")
print(f"  25th percentile:     {length_stats['25th_percentile']:.0f} characters")
print(f"  75th percentile:     {length_stats['75th_percentile']:.0f} characters")

# Individual character length distribution
print(f"\nDetailed Length Distribution (by individual character count):")
print("-" * 60)
length_freq = {}
for length in char_lengths:
    length_freq[length] = length_freq.get(length, 0) + 1

for char_len in sorted(length_freq.keys()):
    count = length_freq[char_len]
    percentage = (count / len(char_lengths)) * 100
    bar_length = int(percentage / 2)  # Create a simple bar visualization
    bar = "█" * bar_length
    print(f"  {char_len:2d} characters: {count:4d} ({percentage:6.2f}%) {bar}")

# Summary by ranges
print(f"\nLength Distribution by Category:")
print("-" * 60)
length_ranges = [
    (1, 3, "Very Short (1-3)"),
    (4, 6, "Short (4-6)"),
    (7, 9, "Medium (7-9)"),
    (10, 12, "Long (10-12)"),
    (13, float('inf'), "Very Long (13+)")
]

for min_len, max_len, label in length_ranges:
    count = sum(1 for l in char_lengths if min_len <= l <= max_len)
    percentage = (count / len(char_lengths)) * 100
    print(f"  {label:20s}: {count:4d} ({percentage:6.2f}%)")

# Show examples by length range
print(f"\nExamples by Length Category:")
print("-" * 60)
for min_len, max_len, label in length_ranges:
    examples = [k for k in stemmed_list if min_len <= len(k) <= max_len]
    if examples:
        print(f"{label}:", ", ".join(sorted(examples)[:8]))
    print()

print("=" * 60)


CHARACTER LENGTH ANALYSIS - STEMMED KEYWORDS
Total unique stemmed keywords: 332

Statistics:
  Mean length:         7.59 characters
  Median length:       7 characters
  Min length:          2 characters
  Max length:          16 characters
  Std Dev:             2.98
  25th percentile:     5 characters
  75th percentile:     9 characters

Detailed Length Distribution (by individual character count):
------------------------------------------------------------
   2 characters:    3 (  0.90%) 
   3 characters:   12 (  3.61%) █
   4 characters:   38 ( 11.45%) █████
   5 characters:   48 ( 14.46%) ███████
   6 characters:   39 ( 11.75%) █████
   7 characters:   27 (  8.13%) ████
   8 characters:   38 ( 11.45%) █████
   9 characters:   44 ( 13.25%) ██████
  10 characters:   33 (  9.94%) ████
  11 characters:   14 (  4.22%) ██
  12 characters:   13 (  3.92%) █
  13 characters:    6 (  1.81%) 
  14 characters:   11 (  3.31%) █
  15 characters:    5 (  1.51%) 
  16 characters:    1 (  0.30%)